# Modelling

## Package Loading

In [1]:
from pickle import dump, load
import os
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.metrics import recall_score, precision_score
from sklearn.ensemble import HistGradientBoostingClassifier
import skops.io as sio

## Data Loading

In [2]:
with open('data/feature_sel/X_train.pkl', 'rb') as file:
    X_train = load(file)

with open('data/feature_sel/y_train.pkl', 'rb') as file:
    y_train = load(file)

with open('data/feature_sel/X_test.pkl', 'rb') as file:
    X_test = load(file)

with open('data/feature_sel/y_test.pkl', 'rb') as file:
    y_test = load(file)

print(f"Training data, row: {len(X_train)}")
display(X_train.head())
display(y_train.head())
print(f"Test data, row: {len(X_test)}")
display(X_test.head())

Training data, row: 7223


,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary
8217,725,1.0,0.0,47,1,104887.43,86622.56
1664,479,1.0,0.0,30,7,143964.36,41879.99
3599,655,0.0,0.0,34,3,0.00,159638.77
5540,684,0.0,1.0,38,4,0.00,75609.84
0,844,0.0,1.0,18,2,160980.03,145936.28


8217    1
1664    0
3599    0
5540    0
0       0
Name: Exited, dtype: int64

Test data, row: 2500


,CreditScore,Geography,Gender,Age,Tenure,Balance,EstimatedSalary
2223,589,0.0,0.0,31,10,110635.32,148218.86
9735,516,2.0,0.0,65,9,102541.10,181490.42
57,582,0.0,1.0,20,4,0.00,55763.66
9243,604,0.0,0.0,56,0,62732.65,124954.56
3428,574,0.0,0.0,34,5,112324.45,17993.43


## Define CV

In [3]:
skf = StratifiedKFold(shuffle=True, random_state=42)
skf.split(X_train, y_train)

<generator object _BaseKFold.split at 0x000001A815D71030>

## Benchmark

Benchmark used here is dummy classifier, assuming all customers would churn.

In [4]:
dummy_mdl = DummyClassifier(strategy='constant',
                            constant=1)

dummy_score = cross_validate(dummy_mdl, X_train, y_train,
                             scoring=['recall', 'precision'],
                             cv=skf)

print(f"Dummy Recall each splits: {dummy_score['test_recall']}")
print(f"Dummy Precision each splits: {dummy_score['test_precision']}")

Dummy Recall each splits: [1. 1. 1. 1. 1.]
Dummy Precision each splits: [0.20207612 0.20207612 0.20207612 0.20221607 0.20221607]


Here we see that, despite "All Churn" benchmark have consistent perfect recall, they also have consistent bad precision. In other words, "All Churn" benchmark will accuse everyone to churn, bringing a lot of false alarm.

In this case, we still are going to optimize recall while also provide precision metric as reality check.

## Fit Model

In [5]:
main_clf = HistGradientBoostingClassifier(max_iter=1000,
                                          categorical_features=['Geography', 'Gender'],
                                          early_stopping=True, ## force early stopping
                                          random_state=42,
                                          class_weight='balanced') ## weighting the class due to imbalanced
                                                                   ## data

main_param = {'learning_rate': [0.01, 0.05, 0.1, 0.12, 0.2, 0.3],
              'max_leaf_nodes': [(2**i)-1 for i in range(2,8)]}

best_main = GridSearchCV(main_clf, main_param,
                         scoring=['recall', 'precision'], ## report both recall and precision
                         refit='recall', ## optimize recall
                         cv=skf)

best_main.fit(X_train, y_train) ## NOTE: this already count as model

print(f"Best Strategy: {best_main.best_params_}")
print(f"Best Recall (cv average): {best_main.best_score_}")

Best Strategy: {'learning_rate': 0.3, 'max_leaf_nodes': 7}
Best Recall (cv average): 0.6835616438356165


In [6]:
best_id = best_main.best_index_
all_recall = [float(best_main.cv_results_['split'+str(i)+'_test_recall'][best_id]) for i in range(0,5)]
all_presis = [float(best_main.cv_results_['split'+str(i)+'_test_precision'][best_id]) for i in range(0,5)]

print(f"Model Recall each splits: {all_recall}")
print(f"Model Precision each splits: {all_presis}")


Model Recall each splits: [0.678082191780822, 0.6712328767123288, 0.6712328767123288, 0.6917808219178082, 0.7054794520547946]
Model Precision each splits: [0.42217484008528783, 0.4224137931034483, 0.38132295719844356, 0.427061310782241, 0.4153225806451613]


Here, we found that 0.68 is the maximum average recall that we can reach, with recall at each CV spread between 0.67 to 0.70. Despite being "worse" than benchmark, we also found improvement on precision: jump from 0.2 on benchmark to at worst 0.38 by the model. Thus the model would raise less false alarms

## Evaluate Model

We already fit our model in the last section. Now we are going to evaluate the performance of our model on test data to see if our model can maintain it's recall while still yield better precision against "all churn".

In [7]:
mdl = best_main.best_estimator_ ## load best model from Tuning object

dummy_mdl.fit(X_train, y_train) ## fit dummy classifier again

dummy_pred = dummy_mdl.predict(X_test)
mdl_pred   = mdl.predict(X_test)

print(f"Dummy Recall on test: {recall_score(y_test, dummy_pred)}")
print(f"Model Recall on test: {recall_score(y_test, mdl_pred)}")
print(f"Dummy Precision on test: {precision_score(y_test, dummy_pred)}")
print(f"Model Precision on test: {precision_score(y_test, mdl_pred)}")

Dummy Recall on test: 1.0
Model Recall on test: 0.6660117878192534
Dummy Precision on test: 0.2036
Model Precision on test: 0.3928157589803013


Here we found that

* Model's recall on test is, unfortunately, worse than any of the cv split's recall
* However, model's precision on test is still better against all churn. Almost twice than all churn's precision on test.

Thus we conclude that the model is good enough against benchmark and have no to less performance fluctuation based on CV, thus the model is ready to serve.

## Save Model

In [8]:
display(mdl)

if not os.path.exists('data/modelling'):
    os.makedirs('data/modelling')

obj_skops = sio.dump(mdl, "data/modelling/model.skops")

,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.3
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",1000
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",7
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.","['Geogr